In [ ]:
import pandas as pd

df = pd.read_csv('../data/IPL_cleaned.csv', parse_dates=['date'])

def create_player_summary(player_name, df):
    player_df = df[df['batter'] == player_name]
    
    if len(player_df) == 0:
        return None
    
    total_runs = player_df['runs_batter'].sum()
    total_matches = player_df['match_id'].nunique()
    total_balls = player_df['balls_faced'].sum()
    
    out_count = df[df['player_out'] == player_name]['match_id'].nunique()
    average = total_runs / out_count if out_count > 0 else total_runs
    strike_rate = (total_runs / total_balls * 100) if total_balls > 0 else 0
    
    # Kis-kis team ke liye khela
    teams_played = df[df['batter'] == player_name]['batting_team'].unique()
    teams_str = ", ".join(teams_played)
    
    # Highest score
    match_scores = player_df.groupby('match_id')['runs_batter'].sum()
    highest_score = match_scores.max()
    
    # Kitne 50+ scores
    fifties = (match_scores >= 50).sum()
    hundreds = (match_scores >= 100).sum()
    
    # Career span
    first_year = player_df['date'].min().year
    last_year = player_df['date'].max().year
    
    summary = (
        f"{player_name} ne IPL mein {first_year} se {last_year} tak khela hai. "
        f"Unhone {teams_str} ke liye khela hai. "
        f"Total {total_matches} matches mein {total_runs} runs banaye hain, "
        f"average {round(average, 2)} aur strike rate {round(strike_rate, 2)} ke sath. "
        f"Unka highest score {highest_score} hai. "
        f"Unke career mein {fifties} fifties aur {hundreds} hundreds hain."
    )
    
    return summary

# Test karo
print(create_player_summary("V Kohli", df))

V Kohli ne IPL mein 2008 se 2025 tak khela hai. Unhone Royal Challengers Bengaluru ke liye khela hai. Total 259 matches mein 8671 runs banaye hain, average 39.59 aur strike rate 132.93 ke sath. Unka highest score 113 hai. Unke career mein 72 fifties aur 8 hundreds hain.


In [ ]:
# Wahi players jo LSTM ke liye qualify hue the (30+ matches)
matches_per_player = df.groupby('batter')['match_id'].nunique()
qualified_players = matches_per_player[matches_per_player >= 30].index.tolist()

print("Total players for RAG:", len(qualified_players))

player_summaries = {}
for player in qualified_players:
    summary = create_player_summary(player, df)
    if summary:
        player_summaries[player] = summary

print("Summaries created:", len(player_summaries))
print("\nSample:")
print(player_summaries[qualified_players[5]])

Total players for RAG: 159
Summaries created: 159

Sample:
AD Mathews ne IPL mein 2009 se 2017 tak khela hai. Unhone Kolkata Knight Riders, Pune Warriors, Delhi Capitals ke liye khela hai. Total 41 matches mein 724 runs banaye hain, average 23.35 aur strike rate 125.91 ke sath. Unka highest score 65 hai. Unke career mein 1 fifties aur 0 hundreds hain.


In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Chroma client banao (local, disk pe save hoga)
client = chromadb.PersistentClient(path="../data/chroma_db")

# Free embedding model use karo
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Collection banao (ya agar already hai toh use karo)
collection = client.get_or_create_collection(
    name="player_summaries",
    embedding_function=embedding_fn
)

# Sab summaries add karo
player_names = list(player_summaries.keys())
summaries_text = list(player_summaries.values())

collection.add(
    documents=summaries_text,
    ids=player_names
)

print("Total documents in collection:", collection.count())

c:\Documents\study\Cricket AI Agent\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
results = collection.query(
    query_texts=["Kohli ke baare mein bata"],
    n_results=2
)

print(results['documents'])
print(results['ids'])

[['MK Lomror ne IPL mein 2018 se 2024 tak khela hai. Unhone Rajasthan Royals, Royal Challengers Bengaluru ke liye khela hai. Total 35 matches mein 527 runs banaye hain, average 18.17 aur strike rate 141.29 ke sath. Unka highest score 54 hai. Unke career mein 1 fifties aur 0 hundreds hain.', 'Dhruv Jurel ne IPL mein 2023 se 2025 tak khela hai. Unhone Rajasthan Royals ke liye khela hai. Total 35 matches mein 680 runs banaye hain, average 28.33 aur strike rate 153.85 ke sath. Unka highest score 70 hai. Unke career mein 4 fifties aur 0 hundreds hain.']]
[['MK Lomror', 'Dhruv Jurel']]


In [ ]:
# Exact naam se try karo
results = collection.query(
    query_texts=["V Kohli"],
    n_results=3
)
print(results['ids'])

# Simple keyword bhi try karo
results2 = collection.query(
    query_texts=["Virat Kohli career runs strike rate"],
    n_results=3
)
print(results2['ids'])

[['V Kohli', 'KV Sharma', 'R Vinay Kumar']]
[['V Kohli', 'Dhruv Jurel', 'KV Sharma']]


In [ ]:
import pandas as pd

master = pd.read_csv('../data/ipl_players_master.csv')
df = pd.read_csv('../data/IPL_cleaned.csv', parse_dates=['date'])

# Hamare dataset se har player ka total runs + debut year nikaalo
our_stats = df.groupby('batter').agg(
    total_runs=('runs_batter', 'sum'),
    debut_year=('season', 'min'),
    last_year=('season', 'max')
).reset_index()

print(our_stats.head())

           batter  total_runs  debut_year  last_year
0  A Ashish Reddy         280        2012       2016
1        A Badoni         963        2022       2025
2      A Chandila           4        2012       2013
3        A Chopra          53        2008       2009
4     A Choudhary          25        2017       2017


In [ ]:
def find_best_match(row, our_stats_df):
    # Similar runs (± 50 runs tolerance) aur overlapping career span wale players dhoondo
    candidates = our_stats_df[
        (abs(our_stats_df['total_runs'] - row['career_runs']) <= 50) &
        (our_stats_df['debut_year'] == row['debut_year'])
    ]
    
    if len(candidates) == 1:
        return candidates.iloc[0]['batter']
    elif len(candidates) > 1:
        # Multiple candidates, sabse close runs wala choose karo
        candidates = candidates.copy()
        candidates['diff'] = abs(candidates['total_runs'] - row['career_runs'])
        return candidates.sort_values('diff').iloc[0]['batter']
    else:
        return None

master['matched_unique_name'] = master.apply(lambda row: find_best_match(row, our_stats), axis=1)

# Kitne successfully match hue
print("Total mapped:", master['matched_unique_name'].notna().sum(), "out of", len(master))

Total mapped: 766 out of 771


In [ ]:
result = master[master['player_name'] == 'Rohit Sharma']
print(result[['player_name', 'career_runs', 'debut_year', 'matched_unique_name']])

    player_name  career_runs  debut_year matched_unique_name
1  Rohit Sharma         7046        2008           RG Sharma


In [ ]:
unmatched = master[master['matched_unique_name'].isna()]
print(unmatched[['player_name', 'career_runs', 'debut_year']])

               player_name  career_runs  debut_year
88           Vijay Shankar         1233        2014
139       Anderson Cummins          612        2014
148         James Faulkner          527        2011
150          Mithun Manhas          514        2008
156  K Nithish Kumar Reddy          485        2023


In [ ]:
# Sirf successfully matched wale rakho, dictionary bana lo
name_mapping = master[master['matched_unique_name'].notna()][['player_name', 'matched_unique_name']]
name_mapping.columns = ['full_name', 'unique_name']

# Lowercase karo taaki case-insensitive lookup ho sake
name_mapping['full_name_lower'] = name_mapping['full_name'].str.lower()

# CSV mein save karo taaki tools.py isko load kar sake
name_mapping.to_csv('../data/player_name_mapping.csv', index=False)

print(name_mapping.head(10))

        full_name     unique_name full_name_lower
0     Virat Kohli         V Kohli     virat kohli
1    Rohit Sharma       RG Sharma    rohit sharma
2  Shikhar Dhawan        S Dhawan  shikhar dhawan
3    David Warner       DA Warner    david warner
4    Suresh Raina        SK Raina    suresh raina
5       M S Dhoni        MS Dhoni       m s dhoni
6       K L Rahul        KL Rahul       k l rahul
7  AB de Villiers  AB de Villiers  ab de villiers
8  Ajinkya Rahane       AM Rahane  ajinkya rahane
9     Chris Gayle        CH Gayle     chris gayle


In [ ]:
import pandas as pd
 
# ---------------------------------------------------------
# Step 1: Data reload karo
# ---------------------------------------------------------
master = pd.read_csv('../data/ipl_players_master.csv')
df = pd.read_csv('../data/IPL_cleaned.csv', parse_dates=['date'])
 
our_stats = df.groupby('batter').agg(
    total_runs=('runs_batter', 'sum'),
    debut_year=('season', 'min'),
    last_year=('season', 'max')
).reset_index()
 

In [ ]:
# ---------------------------------------------------------
# Step 2: Matching function (jo pehle use ki thi)
# ---------------------------------------------------------
def find_best_match(row, our_stats_df):
    candidates = our_stats_df[
        (abs(our_stats_df['total_runs'] - row['career_runs']) <= 50) &
        (our_stats_df['debut_year'] == row['debut_year'])
    ]
    if len(candidates) == 1:
        return candidates.iloc[0]['batter']
    elif len(candidates) > 1:
        candidates = candidates.copy()
        candidates['diff'] = abs(candidates['total_runs'] - row['career_runs'])
        return candidates.sort_values('diff').iloc[0]['batter']
    else:
        return None

In [ ]:
master['matched_unique_name'] = master.apply(lambda row: find_best_match(row, our_stats), axis=1)
 
print("Total mapped:", master['matched_unique_name'].notna().sum(), "out of", len(master))
 
# ---------------------------------------------------------
# Step 3: Suspicious low-run players dhoondo (bowlers jinki
# matching unreliable ho sakti hai kyunki bahut players ke runs similar-low hote hain)
# ---------------------------------------------------------
suspicious = master[(master['career_runs'] < 100) & (master['matched_unique_name'].notna())]
print("\nSuspicious (low-run, possibly-wrong) matches:", len(suspicious))
print(suspicious[['player_name', 'career_runs', 'debut_year', 'matched_unique_name']].to_string())

Total mapped: 766 out of 771

Suspicious (low-run, possibly-wrong) matches: 453
                 player_name  career_runs  debut_year      matched_unique_name
318    Manpreet Singh Grewal           99        2008                  MS Gony
319        Siddharth Chitnis           99        2008                  MS Gony
320       Azmatullah Omarzai           99        2024       Azmatullah Omarzai
321           Michael Clarke           98        2012                MJ Clarke
322          Callum Ferguson           98        2011              CJ Ferguson
323            Prerak Mankad           97        2022                PN Mankad
324              PHKD Mendis           92        2025              PHKD Mendis
325            James Neesham           92        2014              JDS Neesham
326                 P R Shah           92        2008                  PR Shah
327                DJ Jacobs           92        2011                DJ Jacobs
328               Andrew Tye           91        20

In [ ]:
import pandas as pd

master = pd.read_csv('../data/ipl_players_master.csv')
df = pd.read_csv('../data/IPL_cleaned.csv', parse_dates=['date'])

# ---------------------------------------------------------
# Batting fingerprint (runs + debut year) - reliable for career_runs >= 150
# ---------------------------------------------------------
our_batting_stats = df.groupby('batter').agg(
    total_runs=('runs_batter', 'sum'),
    debut_year=('season', 'min'),
    last_year=('season', 'max')
).reset_index()

# ---------------------------------------------------------
# Bowling fingerprint (wickets + debut year) - reliable for bowlers
# ---------------------------------------------------------
NOT_BOWLER_WICKET = ['run out', 'retired hurt', 'retired out', 'obstructing the field']
wickets_df = df[df['wicket_kind'].notna() & (~df['wicket_kind'].isin(NOT_BOWLER_WICKET))]

our_bowling_stats = df.groupby('bowler').agg(
    debut_year=('season', 'min'),
    last_year=('season', 'max')
).reset_index()
wicket_counts = wickets_df.groupby('bowler').size().reset_index(name='total_wickets')
our_bowling_stats = our_bowling_stats.merge(wicket_counts, on='bowler', how='left')
our_bowling_stats['total_wickets'] = our_bowling_stats['total_wickets'].fillna(0)


def find_batting_match(row, stats_df):
    candidates = stats_df[
        (abs(stats_df['total_runs'] - row['career_runs']) <= 30) &
        (stats_df['debut_year'] == row['debut_year'])
    ]
    if len(candidates) == 1:
        return candidates.iloc[0]['batter']
    elif len(candidates) > 1:
        candidates = candidates.copy()
        candidates['diff'] = abs(candidates['total_runs'] - row['career_runs'])
        return candidates.sort_values('diff').iloc[0]['batter']
    return None


def find_bowling_match(row, stats_df):
    if pd.isna(row.get('career_wickets')) or row.get('career_wickets', 0) == 0:
        return None
    candidates = stats_df[
        (abs(stats_df['total_wickets'] - row['career_wickets']) <= 5) &
        (stats_df['debut_year'] == row['debut_year'])
    ]
    if len(candidates) == 1:
        return candidates.iloc[0]['bowler']
    elif len(candidates) > 1:
        candidates = candidates.copy()
        candidates['diff'] = abs(candidates['total_wickets'] - row['career_wickets'])
        return candidates.sort_values('diff').iloc[0]['bowler']
    return None


# ---------------------------------------------------------
# 2-tier strategy:
# - career_runs >= 150 -> trust batting-fingerprint match (reliable)
# - career_runs < 150 (likely bowler/tail-ender) -> use bowling-fingerprint instead
# ---------------------------------------------------------
def resolve_row(row):
    if row['career_runs'] >= 150:
        return find_batting_match(row, our_batting_stats)
    else:
        # Pehle bowling se try karo (zyada reliable inke liye)
        bowling_match = find_bowling_match(row, our_bowling_stats)
        if bowling_match is not None:
            return bowling_match
        # Warna batting se hi try karo (better than nothing)
        return find_batting_match(row, our_batting_stats)


master['matched_unique_name'] = master.apply(resolve_row, axis=1)

print("Total mapped:", master['matched_unique_name'].notna().sum(), "out of", len(master))

# Verify known cases
check_names = ['Jasprit Bumrah', 'Trent Boult', 'Yuzvendra Chahal', 'Rashid Khan', 'Mohammed Shami']
print("\nVerification:")
print(master[master['player_name'].isin(check_names)][['player_name', 'career_wickets', 'debut_year', 'matched_unique_name']])

Total mapped: 764 out of 771

Verification:
          player_name  career_wickets  debut_year matched_unique_name
142       Rashid Khan             158        2017         Rashid Khan
337       Trent Boult             143        2015            TA Boult
338    Mohammed Shami             133        2013      Mohammed Shami
361    Jasprit Bumrah             183        2013           JJ Bumrah
418  Yuzvendra Chahal             221        2013           YS Chahal


In [ ]:
suspicious_v2 = master[(master['career_runs'] < 150) & (master['matched_unique_name'].notna())]
print("Suspicious (should be much fewer now):", len(suspicious_v2))
print(suspicious_v2[['player_name', 'career_runs', 'career_wickets', 'debut_year', 'matched_unique_name']].head(30).to_string())

Suspicious (should be much fewer now): 501
               player_name  career_runs  career_wickets  debut_year matched_unique_name
270             Alex Hales          148               0        2018            AD Hales
271            David Wiese          148              16        2015             D Wiese
272          Shaun Pollock          147              11        2008        Pankaj Singh
273               S Vidyut          145               1        2008               A Nel
274            Sachin Baby          144               2        2013             P Suyal
275                V Nigam          142              11        2025       Ashwani Kumar
276           Marco Jansen          141              36        2021            M Jansen
277              N R SAINI          140               0        2012             N Saini
278          Shubham Dubey          139               0        2024            SB Dubey
279       Anmolpreet Singh          139               0        2021    Anmolp

In [ ]:
name_mapping = master[master['matched_unique_name'].notna()][['player_name', 'matched_unique_name']]
name_mapping.columns = ['full_name', 'unique_name']
name_mapping['full_name_lower'] = name_mapping['full_name'].str.lower()

name_mapping.to_csv('../data/player_name_mapping.csv', index=False)
print("Saved! Total entries:", len(name_mapping))

Saved! Total entries: 764


In [ ]:
import pandas as pd

master = pd.read_csv('../data/ipl_players_master.csv')
df = pd.read_csv('../data/IPL_cleaned.csv', parse_dates=['date'])

# ---------------------------------------------------------
# Batting fingerprint (runs + debut year) - reliable for career_runs >= 150
# ---------------------------------------------------------
our_batting_stats = df.groupby('batter').agg(
    total_runs=('runs_batter', 'sum'),
    debut_year=('season', 'min'),
    last_year=('season', 'max')
).reset_index()

# ---------------------------------------------------------
# Bowling fingerprint (wickets + debut year) - reliable for bowlers
# ---------------------------------------------------------
NOT_BOWLER_WICKET = ['run out', 'retired hurt', 'retired out', 'obstructing the field']
wickets_df = df[df['wicket_kind'].notna() & (~df['wicket_kind'].isin(NOT_BOWLER_WICKET))]

our_bowling_stats = df.groupby('bowler').agg(
    debut_year=('season', 'min'),
    last_year=('season', 'max')
).reset_index()
wicket_counts = wickets_df.groupby('bowler').size().reset_index(name='total_wickets')
our_bowling_stats = our_bowling_stats.merge(wicket_counts, on='bowler', how='left')
our_bowling_stats['total_wickets'] = our_bowling_stats['total_wickets'].fillna(0)


def find_batting_match(row, stats_df):
    # Exact name match ko sabse pehle try karo (sabse reliable)
    exact = stats_df[stats_df['batter'].str.lower() == row['player_name'].lower()]
    if len(exact) == 1:
        return exact.iloc[0]['batter']

    candidates = stats_df[
        (abs(stats_df['total_runs'] - row['career_runs']) <= 15) &
        (stats_df['debut_year'] == row['debut_year'])
    ]
    if len(candidates) == 1:
        return candidates.iloc[0]['batter']
    return None  # Ambiguous - galat guess se accha hai "no match"


def find_bowling_match(row, stats_df):
    if pd.isna(row.get('career_wickets')) or row.get('career_wickets', 0) < 5:
        return None  # Bahut kam wickets - fingerprint reliable nahi hai
    candidates = stats_df[
        (abs(stats_df['total_wickets'] - row['career_wickets']) <= 3) &
        (stats_df['debut_year'] == row['debut_year'])
    ]
    if len(candidates) == 1:
        return candidates.iloc[0]['bowler']
    return None  # Ambiguous - galat guess se accha hai "no match"


# ---------------------------------------------------------
# 2-tier strategy:
# - career_runs >= 150 -> trust batting-fingerprint match (reliable)
# - career_runs < 150 (likely bowler/tail-ender) -> use bowling-fingerprint instead
# ---------------------------------------------------------
def resolve_row(row):
    if row['career_runs'] >= 150:
        return find_batting_match(row, our_batting_stats)
    else:
        # Pehle bowling se try karo (zyada reliable inke liye)
        bowling_match = find_bowling_match(row, our_bowling_stats)
        if bowling_match is not None:
            return bowling_match
        # Warna batting se hi try karo (better than nothing)
        return find_batting_match(row, our_batting_stats)


master['matched_unique_name'] = master.apply(resolve_row, axis=1)

print("Total mapped:", master['matched_unique_name'].notna().sum(), "out of", len(master))

# Verify known cases
check_names = ['Jasprit Bumrah', 'Trent Boult', 'Yuzvendra Chahal', 'Rashid Khan', 'Mohammed Shami']
print("\nVerification:")
print(master[master['player_name'].isin(check_names)][['player_name', 'career_wickets', 'debut_year', 'matched_unique_name']])

Total mapped: 436 out of 771

Verification:
          player_name  career_wickets  debut_year matched_unique_name
142       Rashid Khan             158        2017         Rashid Khan
337       Trent Boult             143        2015            TA Boult
338    Mohammed Shami             133        2013      Mohammed Shami
361    Jasprit Bumrah             183        2013           JJ Bumrah
418  Yuzvendra Chahal             221        2013           YS Chahal


In [ ]:
name_mapping = master[master['matched_unique_name'].notna()][['player_name', 'matched_unique_name']]
name_mapping.columns = ['full_name', 'unique_name']
name_mapping['full_name_lower'] = name_mapping['full_name'].str.lower()

name_mapping.to_csv('../data/player_name_mapping.csv', index=False)
print("Saved! Total entries:", len(name_mapping))

Saved! Total entries: 436
